In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchvision.transforms.functional as tvF
from sklearn import svm
from mpl_toolkits.axes_grid1 import make_axes_locatable

import math

In [2]:
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')


In [3]:
def confusion_cal(y_test, py_test, y_type):
    class_num = len(np.unique(y_test))
    confusion_num = np.zeros((class_num, class_num))
    y_test = torch.tensor(y_test)
    py_test = torch.tensor(py_test)
    if y_type == 0:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(torch.argmax(py_test, dim=1) == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))
    elif y_type == 1:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(py_test == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))

    return confusion_num

In [4]:
dat_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/data'
SPK = []
ptype = []
for rnum in range(0, 11):
    SPK.append(np.load(dat_dir + f'/EndtoEnd/Spk_an_wn_{rnum}.npy'))
    ptype.append(np.load(dat_dir + f'/EndtoEnd/Ptype_an_wn_{rnum}.npy'))

SPK = np.concatenate(SPK, axis=0)
ptype = np.concatenate(ptype, axis=0)

SPK1 = []
ptype1 = []
for rnum in range(0, 32):
    SPK1.append(np.load(dat_dir + f'/EndtoEnd01/Spk_an_wn_{rnum}.npy'))
    ptype1.append(np.load(dat_dir + f'/EndtoEnd01/Ptype_an_wn_{rnum}.npy'))

SPK1 = np.concatenate(SPK1, axis=0)
ptype1 = np.concatenate(ptype1, axis=0)

In [5]:
acu_set = np.zeros((50,5))
acu_set_test = np.zeros((50,5))
for ii in range(0, 50):

    train_ind = np.random.choice(np.size(SPK,axis=0), np.int32(0.8 * np.size(SPK,axis=0)), replace=False)
    test_ind = np.setdiff1d(np.arange(0, np.size(SPK,axis=0)), train_ind)
    test_ind_od = np.random.choice(np.size(SPK1,axis=0), np.int32(0.2 * np.size(SPK1,axis=0)), replace=False)

    x_train = SPK[train_ind,:]
    y_train = ptype[train_ind]

    x_test = SPK[test_ind,:]
    y_test = ptype[test_ind]

    x_test_od = SPK1[test_ind_od,:]
    y_test_od = ptype1[test_ind_od]

    # Initialize model
    model = svm.LinearSVC()

    # Fit the model to training data
    model.fit(x_train, y_train)

    # Check test set accuracy
    acu_set[ii,0] = model.score(x_test, y_test)
    acu_set_test[ii,0] = model.score(x_test_od, y_test_od)

    for jj in range(0,4):
        acu_set[ii,jj + 1] = model.score(x_test[np.argwhere(y_test == jj).flatten()], y_test[np.argwhere(y_test == jj).flatten()])
        acu_set_test[ii,jj + 1] = model.score(x_test_od[np.argwhere(y_test_od == jj).flatten()], y_test_od[np.argwhere(y_test_od == jj).flatten()])



In [6]:
acu_set

array([[0.57782101, 0.50757576, 0.32824427, 0.51282051, 0.94776119],
       [0.57587549, 0.47058824, 0.34328358, 0.58333333, 0.93548387],
       [0.57587549, 0.536     , 0.3826087 , 0.39583333, 0.98461538],
       [0.63035019, 0.63076923, 0.43902439, 0.4516129 , 0.96350365],
       [0.59922179, 0.45669291, 0.42241379, 0.51162791, 0.95070423],
       [0.62256809, 0.6031746 , 0.40650407, 0.46923077, 0.98518519],
       [0.55836576, 0.50420168, 0.34586466, 0.45384615, 0.92424242],
       [0.59727626, 0.55223881, 0.44628099, 0.4469697 , 0.94488189],
       [0.60505837, 0.56521739, 0.43478261, 0.5       , 0.96638655],
       [0.58560311, 0.552     , 0.33587786, 0.484375  , 0.96923077],
       [0.58949416, 0.528     , 0.35658915, 0.51666667, 0.92142857],
       [0.60894942, 0.52857143, 0.40650407, 0.51327434, 0.94927536],
       [0.60894942, 0.56451613, 0.44715447, 0.416     , 0.95774648],
       [0.59338521, 0.56296296, 0.39047619, 0.43884892, 0.94074074],
       [0.58754864, 0.53284672, 0.

In [7]:
save_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/result'
np.save(save_dir + '/svm_endtoend.npy', acu_set)
np.save(save_dir + '/svm_endtoend1.npy', acu_set_test)
np.save(save_dir + '/svm_endtoend_confusion_num.npy', confusion_cal(y_test, model.predict(x_test), y_type=1))
np.save(save_dir + '/svm_endtoend_confusion_num1.npy', confusion_cal(y_test_od, model.predict(x_test_od), y_type=1))

In [8]:
np.round(acu_set_test.mean(axis=0), 3)

array([0.483, 0.394, 0.271, 0.306, 0.968])

In [9]:
np.round(acu_set.mean(axis=0), 3)

array([0.595, 0.552, 0.384, 0.485, 0.952])